# Notebook 05 — Competitor Analysis

**Purpose:**  
Compute head-to-head competitive statistics for every company that participated in
at least one tender where BMMPL also participated.

**Inputs** (`data/processed/`):
- `final_bid_data.xlsx`
- `final_FE.xlsx`
- `company_info.xlsx`

**Output** (`data/processed/`):
- `final_competitor_analysis.xlsx`

**Metrics per company:**

| Column | Definition |
|--------|------------|
| `Total Bids` | Tenders participated in |
| `Bids Won` | Tenders won |
| `Common Bids with BMMPL` | Tenders where both companies participated |
| `Bids Won Against BMMPL` | Of common bids: this company won |
| `Bids Lost Against BMMPL` | Of common bids: BMMPL won |
| `Bids where both lost` | Of common bids: neither won |
| `Overall Winning Probability` | `Bids Won / Total Bids` |
| `Conditional Winning Probability` | `Bids Won Against BMMPL / Common Bids` |
| `Conditional Prob for BMMPL` | `Bids Lost Against BMMPL / Common Bids` |

---

## 1. Imports & Setup

In [1]:
import sys
import re
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from config import PROCESSED_FILES, FOCUS_COMPANY_NAME, FOCUS_COMPANY_ABBR

print(f"Focus company : {FOCUS_COMPANY_NAME}  ({FOCUS_COMPANY_ABBR})")

Focus company : Bhushilp Mines And Minerals Private Limited  (BMMPL)


## 2. Name Normalisation Function

In [2]:
def normalize_name(name: str) -> str:
    """
    Normalise a company name for reliable matching.
    - Strip and uppercase
    - Remove MSE Social Category annotations
    - Remove UNDER PMA suffix
    - Collapse multiple spaces
    """
    if not isinstance(name, str):
        return ""
    s = name.strip().upper()
    s = re.sub(r'\(\s*MSE\s+SOCIAL\s+CATEGORY\s*:[^)]*\)', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\bUNDER\s+PMA\b', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s{2,}', ' ', s).strip()
    return s

## 3. Load Data

In [3]:
bid_data       = pd.read_excel(PROCESSED_FILES['featured_bids'])
financial_eval = pd.read_excel(PROCESSED_FILES['final_fe'])
company_info   = pd.read_excel(PROCESSED_FILES['company_info'])

for df in [bid_data, financial_eval, company_info]:
    df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

# Ensure bid_number is a column
if bid_data.index.name == 'bid_number':
    bid_data = bid_data.reset_index()
if financial_eval.index.name == 'financial_eval_id':
    financial_eval = financial_eval.reset_index()

print(f"bid_data       : {bid_data.shape}")
print(f"financial_eval : {financial_eval.shape}")
print(f"company_info   : {company_info.shape}")

bid_data       : (5689, 16)
financial_eval : (29181, 12)
company_info   : (6832, 5)


In [4]:
# Guard: head-to-head competitor stats key entirely on bid_number sets.
# A non-unique bid_number would corrupt win/loss attribution between companies.
assert bid_data['bid_number'].is_unique, (
    "bid_number is not unique in final_bid_data.xlsx -- check Notebook 02/03 output."
)
print("Guard passed: bid_number is unique.")

Guard passed: bid_number is unique.


## 4. Build Name → company_id Lookup

In [5]:
company_info['norm_name'] = company_info['company_name'].apply(normalize_name)

name_to_id: dict = (
    company_info
    .drop_duplicates(subset='norm_name', keep='first')
    .set_index('norm_name')['company_id']
    .to_dict()
)

# Resolve winner_company_id in bid_data
bid_data['winner_company_id'] = bid_data['winner'].apply(normalize_name).map(name_to_id)

# Resolve focus company id
focus_norm = normalize_name(FOCUS_COMPANY_NAME)
focus_id   = name_to_id.get(focus_norm)

if focus_id is None:
    close = [n for n in name_to_id if FOCUS_COMPANY_ABBR.upper() in n]
    raise ValueError(
        f"Focus company '{FOCUS_COMPANY_NAME}' not found in company_info.\n"
        f"Close matches: {close}"
    )
print(f"{FOCUS_COMPANY_ABBR} company_id : {focus_id}")

BMMPL company_id : CO_00837


## 5. Compute Focus Company Baseline Sets

In [6]:
focus_participated: set = set(
    financial_eval.loc[
        financial_eval['seller_company_id'] == focus_id, 'bid_number'
    ]
)
focus_won: set = set(
    bid_data.loc[bid_data['winner_company_id'] == focus_id, 'bid_number']
)

print(f"{FOCUS_COMPANY_ABBR} — tenders participated : {len(focus_participated)}")
print(f"{FOCUS_COMPANY_ABBR} — tenders won          : {len(focus_won)}")
if len(focus_participated) > 0:
    print(f"{FOCUS_COMPANY_ABBR} — overall win rate     : {len(focus_won)/len(focus_participated)*100:.1f}%")

BMMPL — tenders participated : 43
BMMPL — tenders won          : 12
BMMPL — overall win rate     : 27.9%


## 6. Build Per-Company Win Sets

In [7]:
bid_winner_map: dict = (
    bid_data
    .dropna(subset=['winner_company_id'])
    .groupby('winner_company_id')['bid_number']
    .apply(set)
    .to_dict()
)

## 7. Compute Head-to-Head Statistics

In [8]:
abbr = FOCUS_COMPANY_ABBR

fe_named = financial_eval.merge(
    company_info[['company_id', 'company_name']],
    left_on='seller_company_id', right_on='company_id',
    how='left',
)

records = []

for (comp_id, comp_name), grp in fe_named.groupby(['seller_company_id', 'company_name']):
    participated: set = set(grp['bid_number'])
    won: set          = bid_winner_map.get(comp_id, set())

    total_bids = len(participated)
    bids_won   = len(won & participated)

    common: set   = participated & focus_participated
    common_count  = len(common)

    won_vs_focus  = len(won & common)
    lost_to_focus = len(focus_won & common)
    both_lost     = common_count - won_vs_focus - lost_to_focus

    overall_win  = bids_won      / total_bids   if total_bids   > 0 else 0.0
    cond_win     = won_vs_focus  / common_count if common_count > 0 else 0.0
    cond_focus   = lost_to_focus / common_count if common_count > 0 else 0.0

    records.append({
        'company_id'                          : comp_id,
        'Seller'                              : comp_name,
        'Total Bids'                          : total_bids,
        'Bids Won'                            : bids_won,
        f'Common Bids with {abbr}'            : common_count,
        f'Bids Won Against {abbr}'            : won_vs_focus,
        f'Bids Lost Against {abbr}'           : lost_to_focus,
        'Bids where both lost'                : both_lost,
        'Overall Winning Probability'         : round(overall_win, 4),
        'Conditional Winning Probability'     : round(cond_win,    4),
        f'Conditional Prob for {abbr}'        : round(cond_focus,  4),
    })

competitor_analysis = (
    pd.DataFrame(records)
    .sort_values(f'Common Bids with {abbr}', ascending=False)
    .reset_index(drop=True)
)

print(f"Companies in analysis : {len(competitor_analysis)}")
print(f"With >= 1 common bid  : {(competitor_analysis[f'Common Bids with {abbr}'] > 0).sum()}")

Companies in analysis : 6813
With >= 1 common bid  : 100


## 8. Validate BMMPL Self-Row

In [9]:
bmmpl_row = competitor_analysis.loc[competitor_analysis['company_id'] == focus_id]
assert not bmmpl_row.empty, f"{FOCUS_COMPANY_ABBR} missing from competitor_analysis."

print(f"{FOCUS_COMPANY_ABBR} self-row:")
print(bmmpl_row.T.to_string())

BMMPL self-row:
                                                                           0
company_id                                                          CO_00837
Seller                           Bhushilp Mines And Minerals Private Limited
Total Bids                                                                43
Bids Won                                                                  12
Common Bids with BMMPL                                                    43
Bids Won Against BMMPL                                                    12
Bids Lost Against BMMPL                                                   12
Bids where both lost                                                      19
Overall Winning Probability                                           0.2791
Conditional Winning Probability                                       0.2791
Conditional Prob for BMMPL                                            0.2791


## 9. Top Competitors Preview

In [10]:
abbr = FOCUS_COMPANY_ABBR
top = (
    competitor_analysis[
        (competitor_analysis[f'Common Bids with {abbr}'] > 0) &
        (competitor_analysis['company_id'] != focus_id)
    ]
    .head(15)
    [['Seller', 'Total Bids', 'Bids Won',
      f'Common Bids with {abbr}',
      f'Bids Won Against {abbr}',
      f'Bids Lost Against {abbr}',
      'Overall Winning Probability',
      'Conditional Winning Probability']]
)
print(f"Top 15 competitors by overlap with {abbr}:")
print(top.to_string(index=False))

Top 15 competitors by overlap with BMMPL:
                                                  Seller  Total Bids  Bids Won  Common Bids with BMMPL  Bids Won Against BMMPL  Bids Lost Against BMMPL  Overall Winning Probability  Conditional Winning Probability
Kartikay Exploration And Mining Services Private Limited         127        19                      11                       1                        1                       0.1496                           0.0909
                 South West Pinnacle Exploration Limited          72        19                      11                       2                        2                       0.2639                           0.1818
                       Mining Associates Private Limited         107        16                       9                       1                        4                       0.1495                           0.1111
                                    Mmpl Private Limited          54        12                       9

## 10. Export

In [11]:
competitor_analysis.to_excel(PROCESSED_FILES['competitor_analysis'], index=False)
print(f"Exported competitor_analysis  →  {PROCESSED_FILES['competitor_analysis'].name}  ({len(competitor_analysis)} rows)")
print("\nNotebook 05 complete. Proceed to 06_eda_and_quality.ipynb.")

Exported competitor_analysis  →  final_competitor_analysis.xlsx  (6813 rows)

Notebook 05 complete. Proceed to 06_eda_and_quality.ipynb.
